In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from pandas import Series, DataFrame
from sys import prefix
from listUtils import getFlatList
from musicdb import MusicDBIO
from utils import MusicDBPermDir
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import deezer
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Deezer")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Deezer API(Key=None)
Saving Perminant Deezer DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Deezer


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localRelatedData        = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelatedData".format(db.lower()))
localRelated            = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelated".format(db.lower()))
masterRelatedArtistData = mio.data.getRelatedArtistsData()
localArtistsInfoData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfoData".format(db.lower()))
localArtistsInfo        = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfo".format(db.lower()))
masterArtistsInfoData   = mio.data.getArtistsInfoData()
knownArtists            = mio.data.getSummaryNameData()
errors                  = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Related Artists:       {0}".format(len(localRelated.get())))
print("   Local Related Artists Data:  {0}".format(len(localRelatedData.get())))
print("   Master Related Artists Data: {0}".format(len(masterRelatedArtistData)))
print("   Local Artists Info:          {0}".format(len(localArtistsInfo.get())))
print("   Local Artists Info Data:     {0}".format(len(localArtistsInfoData.get())))
print("   Master Artists Info Data:    {0}".format(masterArtistsInfoData.shape[0]))
print("   Errors:                      {0}".format(len(errors.get())))
print("   Known Summary IDs:           {0}".format(len(knownArtists)))

Deezer Search Results
   Local Related Artists:       93208
   Local Related Artists Data:  0
   Master Related Artists Data: 62431
   Local Artists Info:          313144
   Local Artists Info Data:     0
   Master Artists Info Data:    313129
   Errors:                      1781
   Known Summary IDs:           1069560


# Download Artist Info Data

In [6]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

Deezer API(Key=None)


## Find Artist IDs To Get

In [13]:
artistNames = mio.data.getSummaryNameData()
artistNames.name = "ArtistName"
basicData = DataFrame(artistNames).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
localArtistsInfoDict = localArtistsInfo.get()
artistIDsToGet = basicData[~basicData.index.isin(localArtistsInfoDict.keys())]["ArtistName"]

print("{0} Search Results".format(db))
print("   Known Master Basic Data:   {0}".format(len(artistNames)))
print("   Known Artist Info Names:   {0}".format(len(localArtistsInfoDict)))
print("   Artist Names To Get:       {0}".format(len(artistIDsToGet)))

#   Artist Names To Get:       814603
#   Artist Names To Get:       765878
#   Artist Names To Get:       744853

Deezer Search Results
   Known Master Basic Data:   1069560
   Known Artist Info Names:   334169
   Artist Names To Get:       744853


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "8:00pm")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForLocalArtistsInfo.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalArtistsInfo[artistID] = "NoInfo"
            apiio.sleep(1.5)
        continue
    
    searchedForLocalArtistsInfo[artistID]     = artistName
    searchedForLocalArtistsInfoData[artistID] = response
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting Deezer ArtistIDs]    ==> Time Is 2022-03-20 16:03:20
========================= termTime(day=today,time=8:00pm) =========================
   ====> Terminate Time Set To 2022-03-20 20:00:00 <====
   ====> Will Terminate Process 3 Hours and 56 Minutes From Now
Searching For Artist Info For Tomchess & Ravi Padmanabha (5031920)              	True
Searching For Artist Info For Scarecrow Collection (124160)                     	True
Searching For Artist Info For Ben Goudie-Park (5248621)                         	True
Searching For Artist Info For Susana Santos Silva (4794723)                     	True
Searching For Artist Info For John Silver Jr (128684002)                        	True
Searching For Artist Info For Greg Norwood (54126302)                           	True
Searching For Artist Info For Michel van Den Berg (9722020)                     	True
Searching For Artist Info For Odiseo, Gandulk (1436478)                         	True
Searching For Artist Info Fo

Searching For Artist Info For Billy Randolph & The High Hatters (4236210)       	True
Searching For Artist Info For Nathalie Cardone (360)                            	True
Searching For Artist Info For jean-paul dehavilland,kelly jones,richard potter,darren tansley & nicholas spall (9863260)	True
Searching For Artist Info For Chat Logs (984511)                                	True
Searching For Artist Info For hide and seek DOROTHY (102103062)                 	True
Searching For Artist Info For Johnny Carroll & His Hot Rocks (1368069)          	True
Searching For Artist Info For Dreieck, Aleck Berserk (70513002)                 	True
Searching For Artist Info For RW Carrousel van liefde (124952202)               	True
Searching For Artist Info For Musique Pour Se Détendre (13980941)              	True
Searching For Artist Info For $elfish (13637681)                                	True
Searching For Artist Info For Moebius Effect (8590262)                          	True
Searching For 

Searching For Artist Info For Massimo Di Cataldo (6860)                         	True
Searching For Artist Info For Shella Yolanda (13337511)                         	True
Searching For Artist Info For Horns Of Domination (145452792)                   	True
Searching For Artist Info For Sunchase, NickBee (13552033)                      	True
Searching For Artist Info For Lady G! (85068642)                                	True
Searching For Artist Info For Maldito Mestizo (155298982)                       	True
Searching For Artist Info For Jude Dickinson (156085172)                        	True
Searching For Artist Info For Ramone Hall (7266086)                             	True
Searching For Artist Info For Infamous 1hitta (6770499)                         	True
Searching For Artist Info For Minos (91074962)                                  	True
Searching For Artist Info For Sam Joris & Wout Gooris (91010922)                	True
Searching For Artist Info For Claude Goudimel (1639978

Searching For Artist Info For Midnite Snake (1300778)                           	True
Searching For Artist Info For Dan Savidge (7907620)                             	True
Searching For Artist Info For Midori Karashima (1521489)                        	True
Searching For Artist Info For Mark Sebastian D'Lacey (55944752)                 	True
Searching For Artist Info For Brooks Kerr - Paul Quinichette Quartet (8938670)  	True
Searching For Artist Info For the beige (4636798)                               	True
Searching For Artist Info For Michael Scott (275396)                            	True
Searching For Artist Info For Eleven Tigers (1229220)                           	True
Searching For Artist Info For Carlos Gustavo Alvarez G. (13082233)              	True
Searching For Artist Info For Polyrhythm (1034570)                              	True
Searching For Artist Info For Ddg (128638652)                                   	True
Searching For Artist Info For Cracked Patina (15094082

Searching For Artist Info For 4 or 5 Magicians (303823)                         	True
Searching For Artist Info For Osaka Spa Music (76482752)                        	True
Searching For Artist Info For Umi Yokai (66033242)                              	True
Searching For Artist Info For David M Vines (74046662)                          	True
Searching For Artist Info For Jay Style (566200)                                	True
Searching For Artist Info For Yz Amarr (147018602)                              	True
Searching For Artist Info For The Fractal Impulse (64927702)                    	True
Searching For Artist Info For Chris Liberator & Sterling Moss (6355460)         	True
Searching For Artist Info For Death for Life (86036202)                         	True
Searching For Artist Info For Pawn Shop Poet (6152320)                          	True
Searching For Artist Info For Víctor López & Kevin Romero (79176122)          	True
Searching For Artist Info For Planet Asia & Gold Chain

...

Searching For Artist Info For The Ocean As Mistress (1108900)                   	True
Searching For Artist Info For Aural Hallucinations (90412102)                   	True
Searching For Artist Info For Spaceboy Klakka (11891195)                        	True
Searching For Artist Info For Floike Gloss (114701142)                          	True
Searching For Artist Info For Eugene Jackson As DJ Mindscape (4028016)          	True
Searching For Artist Info For Cuneyt Unsall (10322440)                          	True
Searching For Artist Info For Heat Treated Band (112050192)                     	True
Searching For Artist Info For Jak3 - Trashman (15242517)                        	True
Searching For Artist Info For Rob Ickes & Trey Hensley (7180780)                	True
Searching For Artist Info For Drechsler Steger Tanschek Trio (1331985)          	True
Searching For Artist Info For Primordial Idol (11156452)                        	True
Searching For Artist Info For MR SKWEENZ (76075882)   

Searching For Artist Info For Joseph Indelicato & Chriss Vargas (4333561)       	True
Searching For Artist Info For Mr Postman (13121381)                             	True
Searching For Artist Info For The Same Stream Choir (89788052)                  	True
Searching For Artist Info For Martin Badder & Rowetta (54167092)                	True
Searching For Artist Info For Jules Delsart (1846371)                           	True
Searching For Artist Info For Death Cube K (1336258)                            	True
Searching For Artist Info For Ondřej Pivec (1538202)                           	True
Searching For Artist Info For Stare (11140)                                     	True
Searching For Artist Info For Kopel (176871)                                    	True
Searching For Artist Info For Twitch Music (138005742)                          	True
Searching For Artist Info For Vinyl Junkie & Sanxion (5607340)                  	True
Searching For Artist Info For T & The Wrecks (15635708

Searching For Artist Info For I Musici De Montréal, Yuli Turovsky, Serhiy Salov (96878)	True
Searching For Artist Info For Ryszard Minkiewicz (2741131)                      	True
Searching For Artist Info For Felt (94821752)                                   	True
Searching For Artist Info For Kerry Wilkins (1131716)                           	True
Searching For Artist Info For Orchestra della Radio della Svizzera Italiana (5701825)	True
Searching For Artist Info For Sam Hartshorn & Andy Hartshorn (53442352)         	True
Searching For Artist Info For Nedwed (112129552)                                	True
Searching For Artist Info For Barry Harris Trio (304940)                        	True
Searching For Artist Info For The Miller Stain Limit (4864836)                  	True
Searching For Artist Info For DJ Big E (5509240)                                	True
Searching For Artist Info For Solo Banton, Spectacular (4938481)                	True
Searching For Artist Info For Ok Cougars 

Searching For Artist Info For Brian Daly (4113285)                              	True
Searching For Artist Info For MáScara (78653782)                               	True
Searching For Artist Info For Ali Mulla (3864491)                               	True
Searching For Artist Info For Stinkworx (57471172)                              	True
Searching For Artist Info For Little Leviathan (6132176)                        	True
Searching For Artist Info For Copenhagen Bass (5055431)                         	True
Searching For Artist Info For Goonie Ruffin (91547222)                          	True
Searching For Artist Info For Los Cuatro Pepes (11445286)                       	True
Searching For Artist Info For Nanne & Ankie (72478822)                          	True
Searching For Artist Info For DX (186802)                                       	True
Searching For Artist Info For Return (240904)                                   	True
Searching For Artist Info For Ian Campbell & The Recru

Searching For Artist Info For William "Tycoon" Russ (106128212)                 	True
Searching For Artist Info For Apes on the Moon (91857102)                       	True
Searching For Artist Info For Planet Strychnine (131386102)                     	True
Searching For Artist Info For Triny T (1529876)                                 	True
Searching For Artist Info For Don Reid (4434917)                                	True
Searching For Artist Info For Risto Bsb Nation (102670802)                      	True
Searching For Artist Info For Christopher Moll (5635942)                        	True
Searching For Artist Info For TM$ Woedie (64705732)                             	True
Searching For Artist Info For Northwest Passage Trumpet Trio (101223362)        	True
Searching For Artist Info For Joey M (7721040)                                  	True
Searching For Artist Info For Digidone (5317678)                                	True
Searching For Artist Info For empath records (9074994)

## Save Results

In [10]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [11]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].fillna(0).astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

Found 21025 New Artists
Found 313129 Previous Artists
Found 334154 Total Artists
Found 334154 Unique Artists
Found 334154 Unique + Sorted Artists
Saving Data


In [12]:
localArtistsInfoData.save(data={})

# Download Related Artist Search Data

In [ ]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

## Find Related Artists

In [ ]:
useBasicData       = False
useSelfRelatedData = True
useMasterIDs       = False

if useBasicData is True:
    knownRelatedArtists = localRelated.get()
    basicData = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
    basicData.columns = ["ArtistName", "NumAlbums"]
    artistRelatedToGet = basicData["ArtistName"][~basicData["ArtistName"].index.isin(knownRelatedArtists.keys())]

    print("{0} Search Results".format(db))
    print("   Known Master Basic Data:     {0}".format(basicData.shape[0]))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useSelfRelatedData is True:
    artistRelatedToGet  = {}
    knownRelatedArtists = localRelated.get()
    masterRelatedArtistData = mio.data.getRelatedArtistsData()
    for artistID,artistIDData in masterRelatedArtistData.iteritems():
        artistRelatedToGet.update({str(k): v for k,v in artistIDData.items() if knownRelatedArtists.get(str(k)) is None})
    artistRelatedToGet = Series(artistRelatedToGet)

    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(masterRelatedArtistData)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useMasterIDs is True:
    meu = MusicDBIO()
    mmeDF = meu.getData()
    deezerMatchedArtistsData = mmeDF[mmeDF["DeezerAPI"].notna()][["ArtistName", "DeezerAPI"]]
    deezerMatchedArtists = deezerMatchedArtistsData["ArtistName"].copy(deep=True)
    deezerMatchedArtists.index = deezerMatchedArtistsData["DeezerAPI"]
    artistRelatedToGet = Series({artistID: artistName for artistID,artistName in deezerMatchedArtists.iteritems() if knownRelatedArtists.get(artistID) is None})
    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(deezerMatchedArtists)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "10:15pm")
maxN = 50000000

n  = 0
searchedForLocalRelated         = localRelated.get()
searchedForLocalRelatedData     = localRelatedData.get()
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistRelatedToGet.iteritems()):
    if searchedForLocalRelated.get(artistID) is not None:
        continue

    response = apiio.getArtistRelatedData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(1.5)
        continue
    
    searchedForLocalRelated[artistID]     = artistName
    searchedForLocalRelatedData[artistID] = {str(rec['id']): rec['name'] for rec in response}
    for record in response:
        recID = str(record['id'])
        if searchedForLocalArtistsInfo.get(recID) is None:
            searchedForLocalArtistsInfo[recID]     = artistName
            searchedForLocalArtistsInfoData[recID] = record
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
        localRelated.save(data=searchedForLocalRelated)
        localRelatedData.save(data=searchedForLocalRelatedData)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
    
ts.stop()
print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
localRelated.save(data=searchedForLocalRelated)
localRelatedData.save(data=searchedForLocalRelatedData)
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [ ]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

In [ ]:
df = localRelatedData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = mio.data.getRelatedArtistsData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveRelatedArtistsData(data=searchDF)

In [ ]:
localRelatedData.save(data={})
localArtistsInfoData.save(data={})

# Extra

## Download Data By Artist IDs

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(searchedForArtistIDs.keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = getSearchedForLocalArtistIDsData()
searchedForArtistIDs = getSearchedForLocalArtistIDs()
searchedForErrors = getSearchedForErrors()
for i,dbID in enumerate(artistIDsToGet):
    if searchedForArtistIDs.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        saveSearchedForLocalArtistIDs(searchedForArtistIDs)
        saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
saveSearchedForLocalArtistIDs(searchedForArtistIDs)
saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

# Move Local

In [ ]:
from mdblib.spotify import MusicDBIO
mio = MusicDBIO()

In [ ]:
from mdblib.spotify import moveLocalFiles

In [ ]:
moveLocalFiles()

# Download Album Data

## Find Albums To Get

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(searchedForAlbums.keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "4:00pm")
n  = 0
searchedForAlbums = getSearchedForLocalAlbums()
searchedForErrors = getSearchedForErrors()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        saveSearchedForLocalAlbums(searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

# Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))